# 05.10 · Solutions

[![Open in Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/cloudops-one/training-ai-ml/blob/main/modules/05-powerful-models/10-choosing-and-saving/solutions.ipynb)

## Exercise 1

Best is A at 0.812 with std 0.015, so the bar is 0.797. **A and B** are tied (B at 0.803 ≥ 0.797). C at 0.794 falls just short; D is clearly worse.

- Weekly batch for marketing: strictly by the rule, B (the simpler of the tied pair). But C is only 0.003 below the bar, fully interpretable and cheap; many teams would choose C and say so explicitly. The rule is a guide, not a law.
- Fraud screen worth £50,000 per 0.01 AUC: choose **A**. A 0.018 gap over C is worth about £90,000 a year; explain individual decisions with SHAP.

In [1]:
# --- data helper: works locally and on Colab -------------------------------
from pathlib import Path
import urllib.request

def data_path(name):
    """Return a local path to data/<name>, downloading from GitHub on Colab."""
    for parent in [Path.cwd(), *Path.cwd().parents]:
        candidate = parent / "data" / name
        if candidate.exists():
            return candidate
    url = f"https://raw.githubusercontent.com/cloudops-one/training-ai-ml/main/data/{name}"
    dest = Path(name)
    if not dest.exists():
        urllib.request.urlretrieve(url, dest)
    return dest

import pandas as pd, numpy as np
import matplotlib.pyplot as plt
import warnings; warnings.filterwarnings("ignore")

import sys
sys.path.insert(0, str(data_path("download.py").parent))
from download import load_retail
from sklearn.model_selection import train_test_split

raw = load_retail("Year 2010-2011")
sales = raw[~raw["Invoice"].astype(str).str.startswith("C")]
sales = sales[(sales["Quantity"] > 0) & (sales["Price"] > 0)].dropna(subset=["Customer ID"]).drop_duplicates()
sales = sales.assign(Revenue=sales["Quantity"] * sales["Price"])

cutoff = pd.Timestamp("2011-09-01")                       # features before, target after
past = sales[sales["InvoiceDate"] < cutoff]
future = sales[(sales["InvoiceDate"] >= cutoff) & (sales["InvoiceDate"] < "2011-12-01")]

cust = past.groupby("Customer ID").agg(orders=("Invoice", "nunique"), spend=("Revenue", "sum"),
                                       products=("StockCode", "nunique"),
                                       first_order=("InvoiceDate", "min"), last_order=("InvoiceDate", "max"))
cust["tenure_days"] = (cutoff - cust["first_order"]).dt.days
cust["days_since_last"] = (cutoff - cust["last_order"]).dt.days
future_spend = future.groupby("Customer ID")["Revenue"].sum().reindex(cust.index).fillna(0)
cust["churned"] = (future_spend == 0).astype(int)        # 1 = did not come back
cust = cust.drop(columns=["first_order", "last_order"])

features = ["orders", "spend", "products", "tenure_days", "days_since_last"]
X_train, X_test, y_train, y_test = train_test_split(
    cust[features], cust["churned"], test_size=0.25, random_state=42, stratify=cust["churned"])
print(f"{len(cust):,} customers, churn rate {cust['churned'].mean():.1%}, "
      f"train {len(X_train):,} / test {len(X_test):,}")

def expected_cost(y, p, t, c_voucher=5, c_lost=60):
    """Voucher for everyone with p >= t; a missed churner costs c_lost."""
    y, flag = np.asarray(y), np.asarray(p) >= t
    return c_voucher * flag.sum() + c_lost * ((y == 1) & ~flag).sum()

from pathlib import Path
def models_dir():
    """<repo>/models when run inside the course repo, ./models otherwise (e.g. Colab)."""
    for p in [Path.cwd(), *Path.cwd().parents]:
        if (p / "data" / "download.py").exists():
            d = p / "models"; break
    else:
        d = Path("models")
    d.mkdir(exist_ok=True)
    return d

import time, lightgbm as lgb
from sklearn.pipeline import make_pipeline
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LogisticRegression
from sklearn.neighbors import KNeighborsClassifier
from sklearn.tree import DecisionTreeClassifier
from sklearn.ensemble import RandomForestClassifier
from sklearn.model_selection import StratifiedKFold, cross_validate

cv = StratifiedKFold(5, shuffle=True, random_state=42)
candidates = {
    "logistic regression": (make_pipeline(StandardScaler(), LogisticRegression(max_iter=1000)), "high: one weight per feature"),
    "decision tree (depth 3)": (DecisionTreeClassifier(max_depth=3, random_state=42), "high: 8 readable rules"),
    "KNN (k=51)": (make_pipeline(StandardScaler(), KNeighborsClassifier(51)), "low: 'similar to these customers'"),
    "random forest": (RandomForestClassifier(300, max_depth=6, min_samples_leaf=20, random_state=42, n_jobs=-1), "medium: needs SHAP"),
    "LightGBM (tuned)": (lgb.LGBMClassifier(n_estimators=200, learning_rate=0.01, num_leaves=4,
                                            verbose=-1, random_state=42, n_jobs=1), "medium: needs SHAP"),
}

3,317 customers, churn rate 43.6%, train 2,487 / test 830


## Exercise 2

In [2]:
import joblib, datetime
from sklearn.model_selection import cross_val_predict
from sklearn.metrics import roc_auc_score
gbm = candidates["LightGBM (tuned)"][0]
p_cv = cross_val_predict(gbm, X_train, y_train, cv=cv, method="predict_proba")[:, 1]
grid_t = np.round(np.arange(0.05, 0.96, 0.01), 2)
t_gbm = float(grid_t[int(np.argmin([expected_cost(y_train, p_cv, t) for t in grid_t]))])
gbm.fit(X_train, y_train)
auc_gbm = roc_auc_score(y_test, gbm.predict_proba(X_test)[:, 1])
path = models_dir() / f"churn_lgbm_{datetime.date.today()}_auc{auc_gbm:.3f}.joblib"
joblib.dump({"pipeline": gbm, "features": features, "threshold": t_gbm,
             "metrics": {"test_auc": auc_gbm}}, path)
print(path.name, "threshold", t_gbm)

churn_lgbm_2026-10-28_auc0.739.joblib threshold 0.13


In [3]:
new_customer = pd.DataFrame([{"orders": 2, "spend": 310.50, "products": 14, "tenure_days": 150, "days_since_last": 95}])
b = joblib.load(path)
p = b["pipeline"].predict_proba(new_customer[b["features"]])[0, 1]
print(f"LightGBM: P(churn) = {p:.2f}, voucher: {p >= b['threshold']}")

LightGBM: P(churn) = 0.53, voucher: True


The probabilities differ a little but both models land on the same side of their thresholds for this customer. Expect the two to disagree only near the threshold.

## Exercise 3

In [4]:
def score_customers(path, df):
    bundle = joblib.load(path)
    missing = [f for f in bundle["features"] if f not in df.columns]
    if missing:
        raise ValueError(f"missing feature columns: {missing}")
    p = bundle["pipeline"].predict_proba(df[bundle["features"]])[:, 1]
    return pd.DataFrame({"p_churn": p.round(3), "voucher": p >= bundle["threshold"]}, index=df.index)

print(score_customers(path, X_test.head()))
try:
    score_customers(path, X_test.head().drop(columns="spend"))
except ValueError as e:
    print("error:", e)

             p_churn  voucher
Customer ID                  
14915.0        0.634     True
14193.0        0.634     True
16681.0        0.274     True
17514.0        0.485     True
13345.0        0.634     True
error: missing feature columns: ['spend']
